In [ ]:
import pandas as pd

# 1. Load Level 01
l01 = pd.read_csv('/LEVEL - 01(Section 1 and 1_1).csv')

# 2. Simplified ID Function (Matching your extraction logic)
def create_hhid(row):
    # Convert to numeric first to handle potential floats, then to int, then to string
    fsu = str(int(float(row['FSU_Serial_No'])))
    ssn = str(int(float(row['Second_Stage_Stratum_No'])))
    shn = str(int(float(row['Sample_Household_No'])))
    return f"{fsu}_{ssn}_{shn}"

# Apply the simplified ID
l01['common_id'] = l01.apply(create_hhid, axis=1)

# 3. Create Master DataFrame
master_df = l01[['common_id', 'State', 'Sector', 'District', 'Multiplier']].copy()

# Integrity Check: Ensure unique households
master_df = master_df.drop_duplicates(subset=['common_id'])

print(f"Base Master File created with {len(master_df)} unique households.")
print("Example IDs:", master_df['common_id'].head().values)

Base Master File created with 261953 unique households.
Example IDs: ['46667_2_1' '46667_3_1' '46667_3_2' '46667_3_3' '46667_3_4']


In [ ]:
master_df.head()

,common_id,State,Sector,District,Multiplier
0,46667_2_1,1,1,6,16669
1,46667_3_1,1,1,6,24840
2,46667_3_2,1,1,6,24840
3,46667_3_3,1,1,6,24840
4,46667_3_4,1,1,6,24840


In [ ]:
# --- 1. Load Level 02 ---
l02 = pd.read_csv('/LEVEL - 02 (Section 3).csv')

# Use the exact same ID logic: FSU_Serial_No, Second_Stage_Stratum_No, Sample_Household_No
l02['common_id'] = l02.apply(create_hhid, axis=1)

# --- 2. Identify the Household Head (Relation_to_Head == 1) ---
# We extract key demographic and digital features
head_info = l02[l02['Relation_to_Head'] == 1][[
    'common_id',
    'Age',
    'Gender',
    'Marital_Status',
    'Education_Level',
    'Used_Internet_Last_30_Days'
]].copy()

# Rename columns for clarity in your CLV Master File
head_info = head_info.rename(columns={
    'Age': 'head_age',
    'Gender': 'head_gender',
    'Marital_Status': 'head_marital_status',
    'Education_Level': 'head_education_level',
    'Used_Internet_Last_30_Days': 'is_digital_hh'
})

# --- 3. Calculate Household Size (Total members) ---
hh_size = l02.groupby('common_id').size().reset_index(name='hh_size')

# --- 4. Merge into Master File ---
# Identify which columns already exist to prevent merge duplicates
existing_cols = master_df.columns

# Merge HH Size
if 'hh_size' not in existing_cols:
    master_df = master_df.merge(hh_size, on='common_id', how='left')

# Merge Head Info (filtering for only new columns to avoid _x/_y)
new_head_cols = [c for c in head_info.columns if c not in master_df.columns or c == 'common_id']
master_df = master_df.merge(head_info[new_head_cols], on='common_id', how='left')

# Final cleanup: fill missing demographic info with the most frequent value or 0
master_df['hh_size'] = master_df['hh_size'].fillna(1) # Assume at least 1 member
master_df = master_df.fillna(0)

print("Level 02 features (Demographics & Digital Status) successfully merged.")
print(master_df[['common_id', 'hh_size', 'head_age', 'is_digital_hh']].head())

Level 02 features (Demographics & Digital Status) successfully merged.
   common_id  hh_size  head_age  is_digital_hh
0  46667_2_1        5        54            1.0
1  46667_3_1        4        60            1.0
2  46667_3_2        5        37            2.0
3  46667_3_3        5        34            1.0
4  46667_3_4        3        85            2.0


In [ ]:
print(master_df.head())

   common_id  State  Sector  District  Multiplier  hh_size  head_age  \
0  46667_2_1      1       1         6       16669        5        54   
1  46667_3_1      1       1         6       24840        4        60   
2  46667_3_2      1       1         6       24840        5        37   
3  46667_3_3      1       1         6       24840        5        34   
4  46667_3_4      1       1         6       24840        3        85   

   head_gender  head_marital_status  head_education_level  is_digital_hh  
0            1                    2                     6            1.0  
1            1                    2                    12            1.0  
2            1                    2                     4            2.0  
3            1                    2                     6            1.0  
4            1                    2                    13            2.0  


In [ ]:
# --- 1. Load Level 03 ---
l03 = pd.read_csv('/LEVEL - 03.csv')

# Use the standardized ID logic (FSU_Serial_No + Second_Stage_Stratum_No + Sample_Household_No)
l03['common_id'] = l03.apply(create_hhid, axis=1)

# --- 2. Select and Clean Variables ---
# We use a copy to ensure we don't get SettingWithCopy warnings
l03_features = l03[[
    'common_id',
    'Household_Type',
    'Total_Area_Land_Owned_Acres',
    'Ration_Card_Type',
    'Energy_Source_Cooking'
]].copy()

# --- 3. The Clean Merge ---
# Ensure the key is a string in both dataframes
master_df['common_id'] = master_df['common_id'].astype(str)

# Filter for new columns to avoid duplicate suffix errors (_x, _y)
new_l03_cols = [c for c in l03_features.columns if c not in master_df.columns or c == 'common_id']

# Merge with the Master File
master_df = master_df.merge(l03_features[new_l03_cols], on='common_id', how='left')

# Final Cleanup: Fill missing values with 0 or 'Unknown'
master_df = master_df.fillna(0)

print("Level 03 (Land, Ration, and Cooking Fuel) successfully merged.")
print(master_df[['common_id', 'Household_Type', 'Total_Area_Land_Owned_Acres']].head())

Level 03 (Land, Ration, and Cooking Fuel) successfully merged.
   common_id  Household_Type  Total_Area_Land_Owned_Acres
0  46667_2_1               2                         1.87
1  46667_3_1               4                         0.51
2  46667_3_2               2                         0.52
3  46667_3_3               4                         0.03
4  46667_3_4               9                         0.51


In [ ]:
master_df.head()

,common_id,State,Sector,District,Multiplier,hh_size,head_age,head_gender,head_marital_status,head_education_level,is_digital_hh,Household_Type,Total_Area_Land_Owned_Acres,Ration_Card_Type,Energy_Source_Cooking
0,46667_2_1,1,1,6,16669,5,54,1,2,6,1.0,2,1.87,3,2
1,46667_3_1,1,1,6,24840,4,60,1,2,12,1.0,4,0.51,3,2
2,46667_3_2,1,1,6,24840,5,37,1,2,4,2.0,2,0.52,2,2
3,46667_3_3,1,1,6,24840,5,34,1,2,6,1.0,4,0.03,3,2
4,46667_3_4,1,1,6,24840,3,85,1,2,13,2.0,9,0.51,3,2


In [ ]:
# --- 1. Load and Clean Level 09 ---
l09 = pd.read_csv('/LEVEL - 09 (Section 9  10  11).csv')

# Clean column names (removes hidden spaces or tabs)
l09.columns = l09.columns.str.strip()

# Apply the standardized 3-column ID
l09['common_id'] = l09.apply(create_hhid, axis=1)

# --- 2. Robust Segmenting Logic ---
def segment_l09(code):
    try:
        # We use pd.to_numeric to handle strings, floats, and NaNs safely
        val = pd.to_numeric(code, errors='coerce')
        if pd.isna(val):
            return 'other_l09'

        if 101 <= val <= 499:
            return 'food_spend'
        elif 500 <= val <= 549:
            return 'intoxicant_spend'
        elif 550 <= val <= 599:
            return 'energy_fuel_spend'
        return 'other_l09'
    except:
        return 'other_l09'

# Use the exact column name - stripping it just in case
item_code_col = 'Item_Code_9_1_to_11_4'
value_col = 'Value_Rs_9_1_to_11_4'

l09['l09_category'] = l09[item_code_col].apply(segment_l09)

# --- 3. Aggregate Expenditures (Pivot) ---
l09_split = l09.pivot_table(
    index='common_id',
    columns='l09_category',
    values=value_col,
    aggfunc='sum'
).reset_index().fillna(0)

# --- 4. The "Zero-Fail" Merge ---
# Instead of picking columns, we merge everything EXCEPT 'other_l09' and 'error_code'
cols_to_drop = ['other_l09', 'error_code']
cols_to_merge = [c for c in l09_split.columns if c not in cols_to_drop]

master_df = master_df.merge(l09_split[cols_to_merge], on='common_id', how='left')

# Identify which of our target columns actually made it into master_df
added_cols = [c for c in ['food_spend', 'intoxicant_spend', 'energy_fuel_spend'] if c in master_df.columns]

# Fill only those that exist
master_df[added_cols] = master_df[added_cols].fillna(0)

print(f"Level 09 merge complete. Columns found: {added_cols}")

Level 09 merge complete. Columns found: ['food_spend', 'intoxicant_spend']


In [ ]:
master_df.head()

,common_id,State,Sector,District,Multiplier,hh_size,head_age,head_gender,head_marital_status,head_education_level,...,Household_Type,Total_Area_Land_Owned_Acres,Ration_Card_Type,Energy_Source_Cooking,food_spend_x,intoxicant_spend_x,food_spend_y,intoxicant_spend_y,food_spend,intoxicant_spend
0,46667_2_1,1,1,6,16669,5,54,1,2,6,...,2,1.87,3,2,41536.0,8100.0,41536.0,8100.0,41536.0,8100.0
1,46667_3_1,1,1,6,24840,4,60,1,2,12,...,4,0.51,3,2,10480.0,5600.0,10480.0,5600.0,10480.0,5600.0
2,46667_3_2,1,1,6,24840,5,37,1,2,4,...,2,0.52,2,2,103560.0,4000.0,103560.0,4000.0,103560.0,4000.0
3,46667_3_3,1,1,6,24840,5,34,1,2,6,...,4,0.03,3,2,77480.0,7000.0,77480.0,7000.0,77480.0,7000.0
4,46667_3_4,1,1,6,24840,3,85,1,2,13,...,9,0.51,3,2,51690.0,1000.0,51690.0,1000.0,51690.0,1000.0


In [ ]:
#exporting to csv the master file
master_df.to_csv('master_file_merger_new.csv', index=False)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')